# Train all models (dataset v2) in Google Colab

One run trains every model of the comparison with dataset v2 and 3 seeds (123, 333, 666):

| Model | Input | Learning rate | Batch |
|---|---|---|---|
| `cnn` (from scratch) | `logmel_win1024_v2.npz` | 1e-3 | 32 |
| `linear` (PANNs Cnn14 frozen) | `panns_cnn14_v2.npz` | 1e-3 | 32 |
| `cnn14` (fine-tuning) | `waveform_v2.npz` | 1e-4 | 16 |
| `mobilenetv1` (fine-tuning) | `waveform_v2.npz` | 1e-4 | 16 |

The result is one folder, `MyDrive/tfg/results_v2/`, with `mlflow.db`, `mlruns/` and `models/`.

**Before running:**
1. Upload `logmel_win1024_v2.npz`, `panns_cnn14_v2.npz` and `waveform_v2.npz` from `data/processed/` to `MyDrive/tfg/processed/`.
2. *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

## 1. Settings

In [ ]:
SEEDS = [123, 333, 666]
RUNS = [("cnn", "logmel_win1024_v2.npz", 1e-3, 32),
        ("linear", "panns_cnn14_v2.npz", 1e-3, 32),
        ("cnn14", "waveform_v2.npz", 1e-4, 16),
        ("mobilenetv1", "waveform_v2.npz", 1e-4, 16),]

MANIFEST = "data/splits/manifest_v2.csv"
DRIVE_DIR = "/content/drive/MyDrive/tfg"
REPO_DIR = "/content/acoustic-event-detection"
RESULTS_DIR = "/content/results_v2"
DRIVE_RESULTS = f"{DRIVE_DIR}/results_v2"

## 2. Drive, code and dependencies

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q mlflow==3.17.2 panns_inference==0.1.1
%cd /content
!rm -rf {REPO_DIR}
!git clone -q https://github.com/armofz/acoustic-event-detection.git {REPO_DIR}
!git -C {REPO_DIR} log -1 --format='Code at commit %h: %s'

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Change the runtime type to T4 GPU" 

## 3. Features

The `.npz` files in Drive are seen by the code in its usual place, `data/processed/`.

In [ ]:
import os
os.chdir(REPO_DIR)
!ln -sfn {DRIVE_DIR}/processed data/processed
!ls -lh data/processed/

## 4. Train

Runs already in `MyDrive/tfg/results_v2/models/` are skipped, so if Colab disconnects just *Run all* again.

In [ ]:
import os
from pathlib import Path

# Restore finished runs after a disconnection
!mkdir -p {RESULTS_DIR} {DRIVE_RESULTS}
!cp -r {DRIVE_RESULTS}/. {RESULTS_DIR}/
os.chdir(RESULTS_DIR)

for model, features, learning_rate, batch_size in RUNS:
    for seed in SEEDS:
        run_name = f"{model}_{Path(features).stem}_seed{seed}"
        if Path(f"models/{run_name}.pt").exists():
            print("Already done:", run_name)
            continue
        print(f"\n{run_name}")
        get_ipython().system(
            f"python {REPO_DIR}/src/train.py --model {model} --seed {seed} --manifest {REPO_DIR}/{MANIFEST} "
            f"--features {REPO_DIR}/data/processed/{features} --learning-rate {learning_rate} --batch-size {batch_size}")
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError(f"{run_name} failed")
        !cp -r {RESULTS_DIR}/. {DRIVE_RESULTS}/

## 5. Summary

In [ ]:
import mlflow
mlflow.set_tracking_uri(f"sqlite:///{RESULTS_DIR}/mlflow.db")
runs = mlflow.search_runs(search_all_experiments=True)
cols = ["params.model", "metrics.best_val_f1_macro", "metrics.best_val_accuracy", "metrics.best_val_alert_f1", "metrics.best_val_alert_false_alarms"]
summary = runs[cols].groupby("params.model").agg(["mean", "std"]).round(3)
summary

## 6. Use the results on your computer

1. Download `MyDrive/tfg/results_v2/` (with `mlflow.db`, `mlruns/` and `models/`).
2. MLflow saved absolute paths (`/content/results_v2/mlruns/...`). Inside the folder, run once:
   `python <repo>/src/relocate_mlflow.py`
3. `mlflow ui --backend-store-uri sqlite:///mlflow.db` and open http://127.0.0.1:5000.